# VERA: isolated Kaggle validation

Attach the **v3** private bundle Dataset, enable Internet and GPU T4 x2. The default flow installs a separate runtime, checks its CUDA libraries and validates data/token lengths. It does **not** download Qwen weights or start training. No notebook-kernel restart is needed. The bundled data has 2,000 augmented training conversations but only 86 distinct factual texts: it is a contract bootstrap, not an expert cybersecurity SFT dataset. Full training on it is blocked. Automated checks are not human or legal approval.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

candidates = [
    p.parent for p in Path('/kaggle/input').rglob('bundle.json')
    if (p.parent / 'requirements-kaggle.txt').is_file()
]
if len(candidates) != 1:
    raise RuntimeError('Attach exactly one VERA bundle Dataset; detach older versions.')
BUNDLE = candidates[0]
DATA = BUNDLE / 'data'
WORK = Path('/kaggle/working/vera-finetuning')
WORK.mkdir(parents=True, exist_ok=True)
manifest = json.loads((BUNDLE / 'bundle.json').read_text())
if manifest.get('workflow_version') != 3 or manifest.get('runtime_isolation') != 'venv':
    raise RuntimeError('Attach the v3 isolated-runtime bundle before continuing.')
for relative, expected in manifest['files'].items():
    path = (BUNDLE / relative).resolve()
    if not path.is_relative_to(BUNDLE.resolve()):
        raise RuntimeError(f'Invalid bundle path: {relative}')
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise RuntimeError(f'Bundle verification failed: {relative}')
quality = json.loads((BUNDLE / 'quality-report.json').read_text())
data_hash = hashlib.sha256((DATA / 'manifest.json').read_bytes()).hexdigest()
if quality['passed'] is not True or quality['dataset_sha256'] != data_hash:
    raise RuntimeError('The automated data audit is missing, failed or stale.')
print('Verified package:', BUNDLE)
print('Dataset purpose:', quality['recommended_use'])

## Isolated installation

Packages are installed only under `/kaggle/working/vera-finetuning/runtime`. The installer excludes global/user packages and requires `pip check` to succeed. Continue directly to configuration: do not restart the notebook kernel.

In [ ]:
subprocess.run([
    sys.executable, '-I', str(BUNDLE / 'bootstrap.py'),
    '--bundle', str(BUNDLE), '--work', str(WORK),
], check=True)

In [ ]:
RUNTIME_PYTHON = WORK / 'runtime' / 'bin' / 'python'
marker = WORK / 'runtime' / 'installation.json'
if not marker.is_file() or not RUNTIME_PYTHON.is_file():
    raise RuntimeError('Complete isolated installation first.')
installed = json.loads(marker.read_text())
signature = hashlib.sha256((BUNDLE / 'bundle.json').read_bytes()).hexdigest()
if installed.get('status') != 'ready' or installed.get('bundle_sha256') != signature:
    raise RuntimeError('Installation failed or the bundle changed; rerun the installer.')
GPU_COUNT = 1
MAX_LENGTH = 2048
RUN_BASELINE = False
RUN_SMOKE = False
RUN_FULL_TRAINING = False
RUN_FINAL_TEST = False
EXPORT_FORMAT = None
RESUME_CHECKPOINT = None
if GPU_COUNT not in (1, 2):
    raise ValueError('GPU_COUNT must be 1 or 2.')
SMOKE = WORK / f'smoke-{GPU_COUNT}gpu'
FULL = WORK / f'full-{GPU_COUNT}gpu'
SMOKE_REPORT = SMOKE / 'report.json'
COMMON = ['--data', DATA, '--max-length', MAX_LENGTH]

def run(module, arguments, distributed=False):
    environment = dict(os.environ)
    environment['PYTHONNOUSERSITE'] = '1'
    for name in ('PYTHONPATH', 'PYTHONHOME'):
        environment.pop(name, None)
    environment['CUDA_VISIBLE_DEVICES'] = '0,1' if distributed and GPU_COUNT == 2 else '0'
    command = [str(RUNTIME_PYTHON), '-I', '-m', module]
    if distributed and GPU_COUNT == 2:
        command = [str(RUNTIME_PYTHON), '-I', '-m', 'torch.distributed.run', '--standalone']
        command += ['--nproc_per_node=2', '--module', module]
    subprocess.run(command + [str(item) for item in arguments], env=environment, check=True)

## Runtime checks without Qwen weights

The next cell checks actual library imports, pinned loader/trainer APIs, NumPy interoperability and a small CUDA matrix operation. Failure is recorded and stops execution before model loading. Success does not prove that the 4B model fits or that Qwen training is stable.

In [ ]:
run('vera.training.check_runtime', ['--output', WORK / 'runtime-check.json'])

In [ ]:
run('vera.training.preflight', COMMON + ['--output', WORK / 'preflight.json'])

## Optional baseline, no training

Only after runtime/data checks pass, set `RUN_BASELINE = True` in configuration and run this cell. It downloads the pinned 4B model and evaluates nine stratified validation cases. Keep this disabled if you only want installation diagnostics. Existing reports are never overwritten.

In [ ]:
if RUN_BASELINE:
    runtime_report = json.loads((WORK / 'runtime-check.json').read_text())
    if runtime_report.get('passed') is not True:
        raise RuntimeError('A successful runtime check is required before model loading.')
    run('vera.training.evaluate', COMMON + [
        '--output', WORK / 'baseline-validation.json', '--limit', 9,
        '--max-new-tokens', 768,
    ])
else:
    print('Baseline disabled: no Qwen model weights are downloaded.')

## Optional five-update compatibility experiment

This is disabled by default and is not domain adaptation. It requires an existing baseline and checks finite gradients plus real adapter changes. Enable `RUN_SMOKE` only to investigate training compatibility after the baseline. Do not interpret a successful smoke as evidence of improved cybersecurity quality.

In [ ]:
if RUN_SMOKE:
    if not (WORK / 'baseline-validation.json').is_file():
        raise RuntimeError('Complete the baseline before enabling the smoke experiment.')
    run('vera.training.train', COMMON + ['--output', SMOKE, '--steps', 5], distributed=True)
else:
    print('Smoke training disabled.')

In [ ]:
if RUN_SMOKE:
    run('vera.training.evaluate', COMMON + [
        '--adapter', SMOKE / 'adapter', '--output', WORK / 'smoke-validation.json',
        '--limit', 9, '--max-new-tokens', 768,
    ])

## Full training is not available for the bundled bootstrap

The CLI rejects full training on `contract_bootstrap` data before importing GPU libraries. A separate domain SFT dataset and independent evaluation are still needed. Do not enable full training, final testing or export for this bootstrap.

In [ ]:
if RUN_FULL_TRAINING or RUN_FINAL_TEST or EXPORT_FORMAT is not None:
    raise RuntimeError('The bundled contract-bootstrap data is not approved for domain training.')
print('Full training, final model testing and export remain disabled.')

## Download diagnostics before ending the session

This cell works even when no adapter was trained, or after an earlier cell fails. It includes available runtime/data/baseline reports, provenance and any saved smoke checkpoints. The isolated environment and model download caches are never archived. Do not confuse a diagnostics ZIP with a trained model.

In [ ]:
import zipfile

selected = next((
    path for pattern in ('full-*gpu', 'smoke-*gpu')
    for path in sorted(WORK.glob(pattern), reverse=True)
    if (path / 'adapter').is_dir() or any(path.glob('checkpoint-*'))
), None)
label = selected.name if selected is not None else 'diagnostics'
archive = WORK.parent / f'vera-results-{label}.zip'
files = [WORK / name for name in ('runtime-check.json', 'preflight.json')]
files += list(WORK.glob('*validation.json')) + list(WORK.glob('*test.json'))
if selected is not None:
    files += list((selected / 'adapter').rglob('*'))
    files += [selected / name for name in ('recipe.json', 'report.json', 'environment.txt')]
    for checkpoint in selected.glob('checkpoint-*'):
        files += list(checkpoint.rglob('*'))
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_STORED) as package:
    for path in files:
        if path.is_file():
            package.write(path, str(path.relative_to(WORK)))
    if selected is not None:
        smoke_report = WORK / ('smoke-' + selected.name.split('-', 1)[1]) / 'report.json'
        if smoke_report.is_file():
            package.write(smoke_report, 'provenance/smoke-report.json')
    package.write(DATA / 'manifest.json', 'provenance/data-manifest.json')
    package.write(BUNDLE / 'quality-report.json', 'provenance/quality-report.json')
print('Download:', archive)
print('Contains an adapter or checkpoint:', selected is not None)